# VAR-CLIP Figure 10 visual style study

This notebook is an image-only visual study of transformer-time reference-style injection in VAR-CLIP.

It uses the paper-faithful, training-free SVD Principal Feature Blending (PFB) and Structural Attention Correction (SAC) port from notebook 7, with the following three variants:

1. **PFB** at the paper's third cumulative feature (`F3` / zero-based step `2`)
2. **PFB + SAC** at `F3`
3. **Multi-scale PFB + SAC** at steps `2, 4, 6, 8`, with a decay of `0.7` across injections

Each session uses one style reference from `style_figure10` and six object or scene-only text prompts. There is no style phrase in the prompts, no content image input, no VAE latent mixing, and no metrics in this notebook. All edits happen inside the VAR-CLIP autoregressive transformer path.

Run the notebook in order. The three variant cells are intentionally independent experiment sections and can be rerun after changing only the shared configuration cell.

## 1. Colab setup

Select a GPU runtime before running. The notebook clones the official VAR-CLIP repository and downloads its released checkpoint, the official VAR VAE, and OpenAI CLIP ViT-L/14.

In [ ]:
!nvidia-smi

import os
import subprocess
from pathlib import Path

assert (
    os.path.exists('/usr/local/cuda')
    or os.environ.get('COLAB_GPU')
    or Path('/kaggle/working').exists()
), 'No hosted GPU runtime detected. Select a GPU runtime and reconnect.'

if Path('/kaggle/working').exists():
    RUNTIME_ROOT = Path('/kaggle/working')
elif Path('/content').exists():
    RUNTIME_ROOT = Path('/content')
else:
    RUNTIME_ROOT = Path.cwd()

VAR_CLIP_REPO = 'https://github.com/daixiangzi/VAR-CLIP.git'
VAR_CLIP_DIR = RUNTIME_ROOT / 'VAR-CLIP'
OUTPUT_DIR = RUNTIME_ROOT / 'VAR_CLIP_outputs' / 'paper_faithful_svd_pfb_sac'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not VAR_CLIP_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', VAR_CLIP_REPO, str(VAR_CLIP_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(VAR_CLIP_DIR), 'fetch', 'origin', 'master'], check=True)
    subprocess.run(['git', '-C', str(VAR_CLIP_DIR), 'reset', '--hard', 'origin/master'], check=True)

os.chdir(VAR_CLIP_DIR)
print('VAR-CLIP source:', VAR_CLIP_DIR)
print('outputs:', OUTPUT_DIR)

In [ ]:
!pip -q install gdown huggingface_hub einops typed-argument-parser pytz open_clip_torch pandas tqdm

import gc
import importlib
import math
import random
import sys
import types

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm

assert torch.cuda.is_available(), 'Select a GPU runtime, reconnect, and rerun.'
device = 'cuda'
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from huggingface_hub import hf_hub_download
import gdown

PRETRAINED_DIR = VAR_CLIP_DIR / 'pretrained'
LOCAL_OUTPUT_DIR = VAR_CLIP_DIR / 'local_output'
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vae_path = Path(hf_hub_download(
    repo_id='FoundationVision/var',
    filename='vae_ch160v4096z32.pth',
    local_dir=PRETRAINED_DIR,
))

clip_path = PRETRAINED_DIR / 'ViT-L-14.pt'
clip_url = (
    'https://openaipublic.azureedge.net/clip/models/'
    'b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt'
)
if not clip_path.exists() or clip_path.stat().st_size < 500_000_000:
    subprocess.run(['wget', '-c', '--show-progress', '-O', str(clip_path), clip_url], check=True)

var_clip_path = LOCAL_OUTPUT_DIR / 'ar-ckpt-last.pth'
var_clip_url = 'https://drive.google.com/file/d/10gSxvaKaNKJcnqFhU7hQywU28w3nbgoV/view?usp=sharing'
if not var_clip_path.exists() or var_clip_path.stat().st_size < 100_000_000:
    gdown.download(url=var_clip_url, output=str(var_clip_path), fuzzy=True)

assert vae_path.exists(), vae_path
assert clip_path.exists() and clip_path.stat().st_size > 500_000_000, 'Incomplete CLIP checkpoint.'
assert var_clip_path.exists() and var_clip_path.stat().st_size > 100_000_000, 'Incomplete VAR-CLIP checkpoint.'
print('VAE:', vae_path)
print('CLIP:', clip_path)
print('VAR-CLIP:', var_clip_path)

In [ ]:
# PyTorch 2.6+ needs weights_only=False for the trusted OpenAI TorchScript archive.
clip_source = VAR_CLIP_DIR / 'models' / 'clip.py'
clip_text = clip_source.read_text()
old_call = "pretrained='pretrained/ViT-L-14.pt')"
new_call = "pretrained='pretrained/ViT-L-14.pt', weights_only=False)"
if old_call in clip_text:
    clip_source.write_text(clip_text.replace(old_call, new_call, 1))

setattr(torch.nn.Linear, 'reset_parameters', lambda self: None)
setattr(torch.nn.LayerNorm, 'reset_parameters', lambda self: None)

from clip_util import CLIPWrapper
from models.clip import clip_vit_l14
from tokenizer import tokenize
from models import build_vae_var
from models.basic_var import slow_attn
from models.helpers import sample_with_top_k_top_p_

MODEL_DEPTH = 16
PATCH_NUMS = (1, 2, 3, 4, 5, 6, 8, 10, 13, 16)

vae, var_clip = build_vae_var(
    V=4096,
    Cvae=32,
    ch=160,
    share_quant_resi=4,
    device=device,
    patch_nums=PATCH_NUMS,
    n_cond_embed=768,
    depth=MODEL_DEPTH,
    shared_aln=False,
)

clip_model = CLIPWrapper(clip_vit_l14(pretrained=True).to(device).eval(), normalize=True)
vae.load_state_dict(torch.load(vae_path, map_location='cpu', weights_only=False), strict=True)
checkpoint = torch.load(var_clip_path, map_location='cpu', weights_only=False)
var_clip.load_state_dict(checkpoint['trainer']['var_wo_ddp'], strict=True)
del checkpoint

vae.eval()
var_clip.eval()
for model in (vae, var_clip, clip_model.clip):
    for parameter in model.parameters():
        parameter.requires_grad_(False)

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')
gc.collect()
torch.cuda.empty_cache()
print('Frozen VAR-CLIP-d16, VAR VAE, and CLIP ViT-L/14 are ready.')

## 2. Figure 10 sessions and shared configuration

The selected references and prompts follow the Figure 10 demonstration pattern: one fixed style image is paired with several different object or scene prompts. The prompts contain only semantic content; style is supplied by the reference image.

In [ ]:
STYLE_WORKSPACE_REPO = 'https://github.com/LeeHoang2710/Style-Transfer-Experiment.git'
STYLE_WORKSPACE = RUNTIME_ROOT / 'VAR_Style_Transfer_Workspace'

if not STYLE_WORKSPACE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', STYLE_WORKSPACE_REPO, str(STYLE_WORKSPACE)], check=True)
else:
    subprocess.run(['git', '-C', str(STYLE_WORKSPACE), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(STYLE_WORKSPACE), 'reset', '--hard', 'origin/main'], check=True)

STYLE_FIGURE10_DIR = STYLE_WORKSPACE / 'style_figure10'
if not STYLE_FIGURE10_DIR.exists():
    raise FileNotFoundError(
        f'{STYLE_FIGURE10_DIR} is missing. Add the style_figure10 folder to the '
        'Style-Transfer-Experiment repository before running this notebook.'
    )

OUTPUT_DIR = RUNTIME_ROOT / 'VAR_CLIP_outputs' / 'notebook_08_figure10_visual_style_study'
BASELINE_DIR = OUTPUT_DIR / 'baseline'
VARIANT_DIR = OUTPUT_DIR / 'variants'
AGGREGATE_DIR = OUTPUT_DIR / 'aggregate'
for output_subdir in (BASELINE_DIR, VARIANT_DIR, AGGREGATE_DIR):
    output_subdir.mkdir(parents=True, exist_ok=True)

CFG = 4.0
TOP_K = 600
TOP_P = 0.95
SEED = 42

PAPER_ALPHA = 1.0
PAPER_PFB_FEATURE_INDEX = 2       # F3 in the paper; zero-based VAR step 2
PAPER_SAC_PREDICTION_START = 3    # first prediction that can consume edited F3
MULTISCALE_FEATURE_INDICES = [2, 4, 6, 8]
MULTISCALE_STYLE_DECAY = 0.7

RUN_BASELINE_SECTION = True
RUN_PFB_SECTION = True
RUN_PFB_SAC_SECTION = True
RUN_MULTISCALE_SECTION = True
RUN_AGGREGATE_SECTION = True

FIGURE10_SESSIONS = [
    {
        'name': 'colorful origami reference',
        'style_path': STYLE_FIGURE10_DIR / 'fig10_01_colorful_origami.png',
        'style_label': 'colorful origami',
        'prompts': [
            'a kettle',
            'an armchair',
            'a table',
            'a pair of shoes',
            'a car',
            'a ball',
        ],
    },
    {
        'name': 'flat cartoon vector reference',
        'style_path': STYLE_FIGURE10_DIR / 'fig10_17_flat_cartoon_vector_art.png',
        'style_label': 'flat cartoon vector art',
        'prompts': [
            'a pen',
            'a coral',
            'a bell',
            'a helmet',
            'a bird',
            'a book',
        ],
    },
    {
        'name': 'northern renaissance reference',
        'style_path': STYLE_FIGURE10_DIR / 'fig10_03_northern_renaissance_art.png',
        'style_label': 'northern renaissance art',
        'prompts': [
            'an hourglass',
            'a telescope',
            'a castle with tall spires reflected in water',
            'a piano',
            'a horse running in a field',
            'a train',
        ],
    },
    {
        'name': 'intricate line-art reference',
        'style_path': STYLE_FIGURE10_DIR / 'fig10_11_intricate_line_art_illustration.png',
        'style_label': 'intricate line-art illustration',
        'prompts': [
            'a pine tree',
            'a bear',
            'a cluster of pebbles',
            'a row of mountains',
            'a pair of boots',
        ],
    },
    {
        'name': 'geometric flat reference',
        'style_path': STYLE_FIGURE10_DIR / 'fig10_12_geometric_flat_illustration.png',
        'style_label': 'geometric flat illustration',
        'prompts': [
            'an oval mirror',
            'a leaf',
            'a lemon',
            'a moose',
            'a motorbike',
        ],
    },
    {
        'name': 'metallic 3D reference',
        'style_path': STYLE_FIGURE10_DIR / 'fig10_13_metallic_3d_rendering.png',
        'style_label': 'metallic 3D rendering',
        'prompts': [
            'a deer',
            'a dune crab',
            'a mountain peak',
            'a robotic fish',
            'a drone',
        ],
    },
    {
        'name': 'neon splash comic reference',
        'style_path': STYLE_FIGURE10_DIR / 'fig10_14_neon_splash_comic_art.png',
        'style_label': 'neon splash comic art',
        'prompts': [
            'a boom box',
            'a cube robot',
            'a spaceship',
            'a speeding skier',
            'a jeep car',
        ],
    },
    {
        'name': 'architectural line-art reference',
        'style_path': STYLE_FIGURE10_DIR / 'fig10_15_architectural_line_art.png',
        'style_label': 'architectural line art',
        'prompts': [
            'a chef cooking',
            'a phoenix',
            'a teddy bear',
            'a turtle',
            'an armchair',
        ],
    },
    {
        'name': 'retro sci-fi reference',
        'style_path': STYLE_FIGURE10_DIR / 'fig10_16_retro_sci_fi_graphic.png',
        'style_label': 'retro sci-fi graphic',
        'prompts': [
            'an observatory',
            'a lighthouse',
            'a taxi',
            'a dragon',
            'a domed city on cliffs',
        ],
    },
]

FIGURE10_CASES = []
for session_id, session in enumerate(FIGURE10_SESSIONS):
    assert session['style_path'].exists(), session['style_path']
    for prompt_id, prompt in enumerate(session['prompts']):
        FIGURE10_CASES.append({
            'case_id': f's{session_id + 1:02d}_p{prompt_id + 1:02d}',
            'session_id': session_id,
            'session_name': session['name'],
            'style_path': session['style_path'],
            'style_label': session['style_label'],
            'prompt': prompt,
        })

print(f'Figure 10 sessions: {len(FIGURE10_SESSIONS)}')
print(f'Object/scene prompts: {len(FIGURE10_CASES)}')
print(f'sampling: CFG={CFG}, top-k={TOP_K}, top-p={TOP_P}, seed={SEED}')
for session in FIGURE10_SESSIONS:
    print(f"- {session['name']}: {session['style_path'].name} | {len(session['prompts'])} prompts")


## 3. Exact Principal Feature Blending

For a cumulative feature map $F \in \mathbb{R}^{C\times H\times W}$, flatten spatial dimensions to $F \in \mathbb{R}^{C\times HW}$ and compute

$$F=U\Sigma V^\top,$$

$$\Phi(F)=UW\Sigma V^\top, \qquad W_{ii}=\exp(-i\alpha).$$

At the third accumulated feature:

$$F_3^{gen}\leftarrow\Phi(F_3^{sty})+\left(F_3^{gen}-\Phi(F_3^{gen})\right).$$

No centering, covariance transform, normalization, or learned projection is added; those would change the paper's method.

In [ ]:
def load_reference_image(path, size=256):
    image = Image.open(path).convert('RGB')
    image = ImageOps.fit(image, (size, size), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    image_01 = torchvision.transforms.functional.to_tensor(image).unsqueeze(0).to(device)
    return image_01.mul(2).sub(1), image_01, image


@torch.no_grad()
def extract_multiscale_style_features(image_m11):
    # The released VAR quantizer stores its codebook in float32. Keep VAE
    # extraction out of the surrounding float16 autocast context.
    with torch.autocast(device_type='cuda', enabled=False):
        image_m11 = image_m11.float()
        latent = vae.quant_conv(vae.encoder(image_m11))
        features = vae.quantize.f_to_idxBl_or_fhat(latent, to_fhat=True)
    assert len(features) == len(PATCH_NUMS)
    assert all(feature.shape[-2:] == (PATCH_NUMS[-1], PATCH_NUMS[-1]) for feature in features)
    return [feature.float() for feature in features]


def phi_svd(feature_bchw, alpha=1.0, rank=None):
    '''Paper Eq. (5): exponentially reweighted SVD, applied per sample.'''
    original_dtype = feature_bchw.dtype
    batch, channels, height, width = feature_bchw.shape
    outputs = []

    for batch_id in range(batch):
        matrix = feature_bchw[batch_id].detach().float().reshape(channels, height * width)
        u, singular_values, vh = torch.linalg.svd(matrix, full_matrices=False)
        available_rank = singular_values.numel()
        used_rank = available_rank if rank is None else min(int(rank), available_rank)
        weights = torch.exp(
            -float(alpha) * torch.arange(used_rank, device=matrix.device, dtype=matrix.dtype)
        )
        weighted_s = singular_values[:used_rank] * weights
        reconstructed = (u[:, :used_rank] * weighted_s.unsqueeze(0)) @ vh[:used_rank]
        outputs.append(reconstructed.reshape(channels, height, width))

    return torch.stack(outputs).to(dtype=original_dtype)


def principal_feature_blend(generation_feature, style_feature, alpha=1.0, rank=None, strength=1.0):
    '''PFB with an optional strength multiplier; strength=1 is the paper method.'''
    if generation_feature.shape != style_feature.shape:
        raise ValueError(f'PFB shape mismatch: {generation_feature.shape} vs {style_feature.shape}')
    style_feature = style_feature.to(generation_feature)
    style_component = phi_svd(style_feature, alpha=alpha, rank=rank)
    generation_component = phi_svd(generation_feature, alpha=alpha, rank=rank)
    return generation_feature + float(strength) * (style_component - generation_component)


def apply_feature_edit(generation_feature, style_feature, mode, alpha=1.0, rank=None, strength=1.0):
    if mode == 'none':
        return generation_feature
    if mode == 'replace':
        return style_feature.to(generation_feature)
    if mode == 'pfb':
        return principal_feature_blend(
            generation_feature, style_feature, alpha=alpha, rank=rank, strength=strength
        )
    raise ValueError(f'Unknown edit mode: {mode}')

## 4. Exact Structural Attention Correction

VAR-CLIP has self-attention but no Infinity-style text cross-attention. Text enters through CLIP conditioning and AdaLN. SAC therefore maps cleanly to each VAR self-attention block:

$$Q_s^{gen}\leftarrow Q_s^{con},\qquad K_s^{gen}\leftarrow K_s^{con},$$

while $V_s^{gen}$ remains untouched.

The joint inference batch is ordered as:

`[content_cond, generation_cond, content_uncond, generation_uncond]`.

PFB edits $F_3$ only after residual $R_3$ has been sampled. Because next-scale generation is causal, the first attention computation that can consume edited $F_3$ predicts $R_4$. Thus paper stage `s=3` corresponds to zero-based PFB feature index `2`, while executable SAC begins at zero-based prediction index `3`.

In [ ]:
class SACController:
    def __init__(self, base_batch=1):
        self.base_batch = base_batch
        self.active = False
        self.sac_strength = 1.0
        self.total_calls = 0
        self.max_q_copy_error = 0.0
        self.max_k_copy_error = 0.0

    def reset_statistics(self):
        self.total_calls = 0
        self.max_q_copy_error = 0.0
        self.max_k_copy_error = 0.0


def _sac_attention_forward(attention, x, attn_bias):
    batch4, length, channels = x.shape
    qkv = F.linear(
        x,
        attention.mat_qkv.weight,
        torch.cat((attention.q_bias, attention.zero_k_bias, attention.v_bias)),
    ).view(batch4, length, 3, attention.num_heads, attention.head_dim)

    # Force B,H,L,d layout so Q/K are replaced before K/V cache concatenation.
    q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(dim=0)
    if attention.attn_l2_norm:
        scale_multiplier = attention.scale_mul_1H11.clamp_max(attention.max_scale_mul).exp()
        q = F.normalize(q, dim=-1).mul(scale_multiplier)
        k = F.normalize(k, dim=-1)

    controller = getattr(attention, '_paper_sac_controller', None)
    if controller is not None and controller.active:
        b = controller.base_batch
        if batch4 != 4 * b:
            raise RuntimeError(f'SAC expected joint batch {4 * b}, received {batch4}.')

        # Order: content conditional, generation conditional,
        #        content unconditional, generation unconditional.
        q_content = torch.cat((q[:b], q[:b], q[2*b:3*b], q[2*b:3*b]), dim=0)
        k_content = torch.cat((k[:b], k[:b], k[2*b:3*b], k[2*b:3*b]), dim=0)
        if controller.sac_strength >= 1.0:
            # Direct assignment avoids needless round-off for full SAC.
            q, k = q_content, k_content
        else:
            q = q + float(controller.sac_strength) * (q_content - q)
            k = k + float(controller.sac_strength) * (k_content - k)

        controller.total_calls += 1
        controller.max_q_copy_error = max(
            controller.max_q_copy_error,
            float((q[b:2*b] - q[:b]).abs().max().detach().cpu()),
            float((q[3*b:4*b] - q[2*b:3*b]).abs().max().detach().cpu()),
        )
        controller.max_k_copy_error = max(
            controller.max_k_copy_error,
            float((k[b:2*b] - k[:b]).abs().max().detach().cpu()),
            float((k[3*b:4*b] - k[2*b:3*b]).abs().max().detach().cpu()),
        )

    if attention.caching:
        if attention.cached_k is None:
            attention.cached_k, attention.cached_v = k, v
        else:
            attention.cached_k = torch.cat((attention.cached_k, k), dim=2)
            attention.cached_v = torch.cat((attention.cached_v, v), dim=2)
        k, v = attention.cached_k, attention.cached_v

    output = slow_attn(
        query=q,
        key=k,
        value=v,
        scale=attention.scale,
        attn_mask=attn_bias,
        dropout_p=attention.attn_drop if attention.training else 0.0,
    ).transpose(1, 2).reshape(batch4, length, channels)
    return attention.proj_drop(attention.proj(output))


class PaperSACPatch:
    def __init__(self, model, controller):
        self.model = model
        self.controller = controller
        self.original_forwards = []

    def __enter__(self):
        for block in self.model.blocks:
            attention = block.attn
            self.original_forwards.append((attention, attention.forward))
            attention._paper_sac_controller = self.controller
            attention.forward = types.MethodType(_sac_attention_forward, attention)
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        for attention, original_forward in self.original_forwards:
            attention.forward = original_forward
            if hasattr(attention, '_paper_sac_controller'):
                delattr(attention, '_paper_sac_controller')
        return False

## 5. Joint dual-path autoregressive inference

This function implements Algorithm 1 on VAR-CLIP. Both paths share prompt, seed, and random draws. They are exactly identical through $F_3$; only PFB creates divergence. SAC then restores content-path attention geometry during later refinement.

In [ ]:
def prepare_prompt_embedding(prompt):
    tokens = tokenize([prompt]).to(device)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        return clip_model.encode_text(tokens)


def _sample_stream(logits, rng, top_k, top_p):
    return sample_with_top_k_top_p_(
        logits.clone(), rng=rng, top_k=top_k, top_p=top_p, num_samples=1
    )[:, :, 0]


@torch.no_grad()
def paper_dual_path_generate(
    model,
    prompt_embedding,
    style_features,
    *,
    seed=42,
    cfg=1.5,
    top_k=500,
    top_p=0.95,
    pfb_feature_index=2,
    pfb_feature_indices=None,
    sac_prediction_start=3,
    edit_mode='pfb',
    alpha=1.0,
    rank=None,
    style_strength=1.0,
    style_decay=1.0,
    sac_strength=1.0,
    enable_sac=True,
):
    '''Paper Algorithm 1 adapted to VAR-CLIP's 10-stage CFG inference.'''
    base_batch = prompt_embedding.shape[0]
    if base_batch != 1:
        raise ValueError('Demo implementation currently supports one prompt per call.')
    if not 0 <= pfb_feature_index < len(model.patch_nums):
        raise ValueError('Invalid PFB feature index.')
    if enable_sac and not 0 <= sac_prediction_start < len(model.patch_nums):
        raise ValueError('Invalid SAC prediction start.')
    if pfb_feature_indices is None:
        pfb_feature_indices = [pfb_feature_index]
    else:
        pfb_feature_indices = sorted(set(int(index) for index in pfb_feature_indices))
    if not pfb_feature_indices or any(index < 0 or index >= len(model.patch_nums) for index in pfb_feature_indices):
        raise ValueError('Invalid PFB feature indices.')
    if not 0.0 <= float(sac_strength) <= 1.0:
        raise ValueError('sac_strength must be between 0 and 1.')

    model.eval()
    content_rng = torch.Generator(device=device).manual_seed(seed)
    generation_rng = torch.Generator(device=device).manual_seed(seed)

    null_embedding = model.noise(torch.tensor(0, device=device)).unsqueeze(0).expand(base_batch, -1)
    joint_condition = model.cond_proj(torch.cat((
        prompt_embedding,
        prompt_embedding,
        null_embedding,
        null_embedding,
    ), dim=0))

    level_position = model.lvl_embed(model.lvl_1L) + model.pos_1LC
    next_tokens = (
        joint_condition.unsqueeze(1).expand(4 * base_batch, model.first_l, -1)
        + model.pos_start.expand(4 * base_batch, model.first_l, -1)
        + level_position[:, :model.first_l]
    )

    content_fhat = joint_condition.new_zeros(
        base_batch, model.Cvae, model.patch_nums[-1], model.patch_nums[-1]
    )
    generation_fhat = torch.zeros_like(content_fhat)
    content_trace, generation_trace = [], []

    controller = SACController(base_batch)
    controller.sac_strength = float(sac_strength)
    controller.reset_statistics()
    for block in model.blocks:
        block.attn.kv_caching(True)

    current_length = 0
    pre_pfb_max_difference = 0.0
    pfb_relative_change_by_step = {}
    try:
        with PaperSACPatch(model, controller):
            for step_id, patch_num in enumerate(model.patch_nums):
                controller.active = enable_sac and step_id >= sac_prediction_start
                current_length += patch_num * patch_num

                condition_for_blocks = model.shared_ada_lin(joint_condition)
                hidden = next_tokens
                for block in model.blocks:
                    hidden = block(x=hidden, cond_BD=condition_for_blocks, attn_bias=None)
                all_logits = model.get_logits(hidden, joint_condition)

                ratio = step_id / model.num_stages_minus_1
                cfg_ratio = cfg * ratio
                content_logits = (1 + cfg_ratio) * all_logits[:base_batch] - cfg_ratio * all_logits[2*base_batch:3*base_batch]
                generation_logits = (1 + cfg_ratio) * all_logits[base_batch:2*base_batch] - cfg_ratio * all_logits[3*base_batch:4*base_batch]

                content_indices = _sample_stream(content_logits, content_rng, top_k, top_p)
                generation_indices = _sample_stream(generation_logits, generation_rng, top_k, top_p)

                content_residual = model.vae_quant_proxy[0].embedding(content_indices).transpose(1, 2).reshape(
                    base_batch, model.Cvae, patch_num, patch_num
                )
                generation_residual = model.vae_quant_proxy[0].embedding(generation_indices).transpose(1, 2).reshape(
                    base_batch, model.Cvae, patch_num, patch_num
                )

                content_fhat, content_next = model.vae_quant_proxy[0].get_next_autoregressive_input(
                    step_id, len(model.patch_nums), content_fhat, content_residual
                )
                generation_fhat, generation_next = model.vae_quant_proxy[0].get_next_autoregressive_input(
                    step_id, len(model.patch_nums), generation_fhat, generation_residual
                )

                if step_id < min(pfb_feature_indices):
                    pre_pfb_max_difference = max(
                        pre_pfb_max_difference,
                        float((content_fhat - generation_fhat).abs().max().detach().cpu()),
                    )

                if step_id in pfb_feature_indices and edit_mode != 'none':
                    injection_order = pfb_feature_indices.index(step_id)
                    effective_strength = float(style_strength) * float(style_decay) ** injection_order
                    generation_before_edit = generation_fhat.clone()
                    generation_fhat = apply_feature_edit(
                        generation_fhat,
                        style_features[step_id],
                        mode=edit_mode,
                        alpha=alpha,
                        rank=rank,
                        strength=effective_strength,
                    )
                    pfb_relative_change_by_step[step_id] = float(
                        (generation_fhat - generation_before_edit).norm()
                        / generation_before_edit.norm().clamp_min(1e-8)
                    )
                    if step_id != model.num_stages_minus_1:
                        next_patch = model.patch_nums[step_id + 1]
                        generation_next = F.interpolate(
                            generation_fhat, size=(next_patch, next_patch), mode='area'
                        )

                content_trace.append(content_fhat.detach().clone())
                generation_trace.append(generation_fhat.detach().clone())

                if step_id != model.num_stages_minus_1:
                    next_patch = model.patch_nums[step_id + 1]

                    content_tokens = model.word_embed(
                        content_next.view(base_batch, model.Cvae, -1).transpose(1, 2)
                    ) + level_position[:, current_length:current_length + next_patch ** 2]
                    generation_tokens = model.word_embed(
                        generation_next.view(base_batch, model.Cvae, -1).transpose(1, 2)
                    ) + level_position[:, current_length:current_length + next_patch ** 2]

                    next_tokens = torch.cat((
                        content_tokens,
                        generation_tokens,
                        content_tokens,
                        generation_tokens,
                    ), dim=0)

        content_image = model.vae_proxy[0].fhat_to_img(content_fhat).add(1).mul(0.5)
        generation_image = model.vae_proxy[0].fhat_to_img(generation_fhat).add(1).mul(0.5)
        return {
            'content_image_01': content_image,
            'stylized_image_01': generation_image,
            'content_features': content_trace,
            'generation_features': generation_trace,
            'pre_pfb_max_difference': pre_pfb_max_difference,
            'sac_calls': controller.total_calls,
            'max_q_copy_error': controller.max_q_copy_error,
            'max_k_copy_error': controller.max_k_copy_error,
            'pfb_relative_change_by_step': pfb_relative_change_by_step,
        }
    finally:
        controller.active = False
        for block in model.blocks:
            block.attn.kv_caching(False)

## 6. Shared image-only experiment helpers

The helpers below save every image immediately. This keeps the three experiment sections rerunnable and lets the aggregate section work from the saved outputs. No numerical evaluation is computed.

In [ ]:
def _safe_name(text):
    return ''.join(character if character.isalnum() else '_' for character in str(text)).strip('_').lower()


def _image_tensor_to_pil(image):
    tensor = image.detach().float().cpu()
    if tensor.ndim == 4:
        tensor = tensor[0]
    array = tensor.clamp(0, 1).permute(1, 2, 0).mul(255).byte().numpy()
    return Image.fromarray(array)


def _save_image_tensor(image, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    _image_tensor_to_pil(image).save(path)


STYLE_IMAGE_CACHE = {}
STYLE_FEATURE_CACHE = {}


def get_style_image(style_path):
    key = str(Path(style_path))
    if key not in STYLE_IMAGE_CACHE:
        _, style_01, _ = load_reference_image(style_path)
        STYLE_IMAGE_CACHE[key] = style_01.detach().float().cpu()
    return STYLE_IMAGE_CACHE[key]


def get_style_features(style_path):
    key = str(Path(style_path))
    if key not in STYLE_FEATURE_CACHE:
        style_m11, _, _ = load_reference_image(style_path)
        with torch.inference_mode():
            features = extract_multiscale_style_features(style_m11)
        STYLE_FEATURE_CACHE[key] = [feature.detach().float().cpu() for feature in features]
        del style_m11, features
        gc.collect()
        torch.cuda.empty_cache()
    return [feature.to(device) for feature in STYLE_FEATURE_CACHE[key]]


def generate_content_image(prompt):
    prompt_embedding = prepare_prompt_embedding(prompt)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        result = paper_dual_path_generate(
            var_clip,
            prompt_embedding,
            [],
            seed=SEED,
            cfg=CFG,
            top_k=TOP_K,
            top_p=TOP_P,
            pfb_feature_index=PAPER_PFB_FEATURE_INDEX,
            sac_prediction_start=PAPER_SAC_PREDICTION_START,
            edit_mode='none',
            enable_sac=False,
        )
    image = result['content_image_01'].detach().float().cpu()
    del prompt_embedding, result
    gc.collect()
    torch.cuda.empty_cache()
    return image


def generate_variant_image(prompt, style_path, *, pfb_feature_indices, style_decay, enable_sac):
    prompt_embedding = prepare_prompt_embedding(prompt)
    style_features = get_style_features(style_path)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        result = paper_dual_path_generate(
            var_clip,
            prompt_embedding,
            style_features,
            seed=SEED,
            cfg=CFG,
            top_k=TOP_K,
            top_p=TOP_P,
            pfb_feature_index=PAPER_PFB_FEATURE_INDEX,
            pfb_feature_indices=pfb_feature_indices,
            sac_prediction_start=PAPER_SAC_PREDICTION_START,
            edit_mode='pfb',
            alpha=PAPER_ALPHA,
            rank=None,
            style_strength=1.0,
            style_decay=style_decay,
            sac_strength=1.0,
            enable_sac=enable_sac,
        )
    image = result['stylized_image_01'].detach().float().cpu()
    del prompt_embedding, style_features, result
    gc.collect()
    torch.cuda.empty_cache()
    return image


def _plot_image(axis, image):
    axis.imshow(image[0].detach().float().clamp(0, 1).permute(1, 2, 0).numpy())
    axis.axis('off')


def _show_variant_session(session, case_images, variant_name, save_path):
    columns = ['style reference'] + [case['prompt'] for case in FIGURE10_CASES if case['session_id'] == session['session_id']]
    figure, axes = plt.subplots(1, len(columns), figsize=(3.2 * len(columns), 4.2), squeeze=False)
    axes = axes[0]
    _plot_image(axes[0], get_style_image(session['style_path']))
    axes[0].set_title(f"style reference\n{session['style_label']}", fontsize=10)
    session_cases = [case for case in FIGURE10_CASES if case['session_id'] == session['session_id']]
    for column_id, case in enumerate(session_cases, start=1):
        _plot_image(axes[column_id], case_images[case['case_id']])
        axes[column_id].set_title(f'"{case["prompt"]}"', fontsize=10)
    figure.suptitle(f'{variant_name} | {session["name"]}', fontsize=14)
    figure.tight_layout()
    figure.savefig(save_path, dpi=180, bbox_inches='tight')
    print('saved:', save_path)
    plt.show()
    plt.close(figure)


def run_variant_section(variant_name, *, pfb_feature_indices, style_decay, enable_sac):
    section_dir = VARIANT_DIR / _safe_name(variant_name)
    section_dir.mkdir(parents=True, exist_ok=True)
    results = {}
    for case in tqdm(FIGURE10_CASES, desc=variant_name):
        image = generate_variant_image(
            case['prompt'],
            case['style_path'],
            pfb_feature_indices=pfb_feature_indices,
            style_decay=style_decay,
            enable_sac=enable_sac,
        )
        results[case['case_id']] = image
        _save_image_tensor(image, section_dir / f"{case['case_id']}.png")

    for session in FIGURE10_SESSIONS:
        gallery_path = section_dir / f"session_{session['session_id'] + 1:02d}.png"
        _show_variant_session(session, results, variant_name, gallery_path)
    return results


## 7. Baseline content generation

This section generates the native VAR-CLIP image for every object or scene prompt using `CFG=4.0`, `top-k=600`, and `top-p=0.95`. The reference style image is not used in this section. These baseline images are reused in the final visual comparison.

In [ ]:
BASELINE_RESULTS = {}

if RUN_BASELINE_SECTION:
    for case in tqdm(FIGURE10_CASES, desc='VAR-CLIP baseline content images'):
        image = generate_content_image(case['prompt'])
        BASELINE_RESULTS[case['case_id']] = image
        _save_image_tensor(image, BASELINE_DIR / f"{case['case_id']}.png")

print(f'baseline images available: {len(BASELINE_RESULTS)}')


## 8. Variant 1: PFB

Only the paper's single PFB intervention is active. Style is injected into the cumulative feature at `F3` / zero-based step `2`; SAC is disabled.

In [ ]:
PFB_RESULTS = {}

if RUN_PFB_SECTION:
    PFB_RESULTS = run_variant_section(
        'PFB',
        pfb_feature_indices=[PAPER_PFB_FEATURE_INDEX],
        style_decay=1.0,
        enable_sac=False,
    )


## 9. Variant 2: PFB + SAC

This is the paper-style single-step intervention plus Structural Attention Correction. SAC copies the content-path self-attention queries and keys into the generation path after the edited `F3` becomes causally available; generation-path values remain unchanged.

In [ ]:
PFB_SAC_RESULTS = {}

if RUN_PFB_SAC_SECTION:
    PFB_SAC_RESULTS = run_variant_section(
        'PFB + SAC',
        pfb_feature_indices=[PAPER_PFB_FEATURE_INDEX],
        style_decay=1.0,
        enable_sac=True,
    )


## 10. Variant 3: Multi-scale PFB + SAC with decay

This variant repeats the same transformer-time PFB edit at VAR steps `2, 4, 6, 8`. The style strength is `1.0, 0.7, 0.49, 0.343` across those injections, while SAC remains active from the first prediction after `F3`.

In [ ]:
MULTISCALE_RESULTS = {}

if RUN_MULTISCALE_SECTION:
    MULTISCALE_RESULTS = run_variant_section(
        'Multi-scale PFB + SAC (decay=0.7)',
        pfb_feature_indices=MULTISCALE_FEATURE_INDICES,
        style_decay=MULTISCALE_STYLE_DECAY,
        enable_sac=True,
    )


## 11. Final aggregate comparison

Each row contains the fixed style reference, the text prompt, the native VAR-CLIP baseline, and the three transformer-time style variants. This section only creates visual grids; it does not compute metrics.

In [ ]:
def _plot_prompt_panel(axis, prompt):
    axis.set_facecolor('white')
    axis.text(0.5, 0.5, f'"{prompt}"', ha='center', va='center', wrap=True, fontsize=12)
    axis.set_xticks([])
    axis.set_yticks([])
    for spine in axis.spines.values():
        spine.set_color('black')
        spine.set_linewidth(1.0)


def _show_aggregate_session(session, save_path):
    session_cases = [case for case in FIGURE10_CASES if case['session_id'] == session['session_id']]
    columns = [
        'style reference',
        'text prompt',
        'VAR-CLIP baseline',
        'PFB',
        'PFB + SAC',
        'Multi-scale PFB + SAC\n(decay=0.7)',
    ]
    figure, axes = plt.subplots(
        len(session_cases), len(columns),
        figsize=(3.2 * len(columns), 3.5 * len(session_cases)),
        squeeze=False,
    )
    for row_id, case in enumerate(session_cases):
        row_images = [
            get_style_image(session['style_path']),
            None,
            BASELINE_RESULTS[case['case_id']],
            PFB_RESULTS[case['case_id']],
            PFB_SAC_RESULTS[case['case_id']],
            MULTISCALE_RESULTS[case['case_id']],
        ]
        for column_id, (column, image) in enumerate(zip(columns, row_images)):
            axis = axes[row_id, column_id]
            if image is None:
                _plot_prompt_panel(axis, case['prompt'])
            else:
                _plot_image(axis, image)
            if row_id == 0:
                axis.set_title(column, fontsize=10)
    figure.suptitle(f'Figure 10 visual comparison | {session["name"]}', fontsize=14)
    figure.tight_layout()
    figure.savefig(save_path, dpi=180, bbox_inches='tight')
    print('saved:', save_path)
    plt.show()
    plt.close(figure)


if RUN_AGGREGATE_SECTION:
    missing = {
        'baseline': len(BASELINE_RESULTS),
        'PFB': len(PFB_RESULTS),
        'PFB + SAC': len(PFB_SAC_RESULTS),
        'Multi-scale PFB + SAC': len(MULTISCALE_RESULTS),
    }
    expected = len(FIGURE10_CASES)
    if any(count != expected for count in missing.values()):
        raise RuntimeError(f'Run the baseline and all three variant sections first: {missing}, expected {expected}.')
    for session in FIGURE10_SESSIONS:
        _show_aggregate_session(
            session,
            AGGREGATE_DIR / f"session_{session['session_id'] + 1:02d}_aggregate.png",
        )


## Output layout

The notebook writes only PNG images under:

`/content/VAR_CLIP_outputs/notebook_08_figure10_visual_style_study/`

The `baseline/` folder contains native VAR-CLIP outputs, `variants/` contains one folder per method plus its six-prompt session galleries, and `aggregate/` contains the final row-wise comparisons.